<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 28 · Simulation of Financial Models

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

import datetime as dt
import math
from dataclasses import dataclass
from typing import Any, Protocol

import numpy as np
import pandas as pd

from dxlib import *


class Pricer(Protocol):
    def value(self, spot: float) -> tuple[float, float]: ...


snapshot_path = PROJECT_ROOT / 'data' / 'spx_options_snapshot.csv'
snap = load_spx_snapshot(snapshot_path) if snapshot_path.exists() else None
if snap is not None:
    short_expiry = dt.date(2026, 3, 20)
    long_expiries = [dt.date(2026, 6, 18), dt.date(2026, 12, 18)]
    short_surface_df = select_small_surface(snap, [short_expiry], rate=0.03)
    long_surface_df = select_small_surface(snap, long_expiries, rate=0.03)
    surface_df = long_surface_df
    paths = 15_000
    steps_per_year = 320
    rate = 0.03
    h_params = HestonParams(
        kappa=2.6,
        theta=0.046,
        vol_of_vol=0.9,
        rho=-0.67,
        v0=0.018,
    )
    params = h_params
    local_map = {
        dt.date(2026, 6, 18): (0.0606, 0.0128, -0.732),
        dt.date(2026, 12, 18): (0.0484, 0.0149, -0.763),
    }
    seed = 11


## Simulation as an Interface Contract

Execute the code examples below.


## Random Numbers and Variance Reduction

Execute the code examples below.


In [ ]:
import numpy as np


def standard_normals(
    shape,
    *,
    seed=None,
    antithetic=False,
    moment_matching=False,
):
    rng = np.random.default_rng(seed=seed)
    out = rng.standard_normal(size=shape).astype(float, copy=False)

    if antithetic:
        last = shape[-1]
        half = last // 2
        out[..., half:] = -out[..., :half]

    if moment_matching:
        out = (out - out.mean()) / out.std(ddof=0)

    return out

## Time Grids and Step Sizes

Execute the code examples below.


## Geometric Brownian Motion (GBM)

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class GeometricBrownianMotion:
    drift: float
    volatility: float
    seed: int | None = None
    antithetic: bool = True
    moment_matching: bool = True

    def simulate_paths(self, spot, time_grid, paths):
        step_sizes = np.diff(time_grid)
        shocks = standard_normals(
            (step_sizes.size, paths),
            seed=self.seed,
            antithetic=self.antithetic,
            moment_matching=self.moment_matching,
        )
        out = np.empty((paths, time_grid.size), dtype=float)
        out[:, 0] = spot
        for step_index, dt in enumerate(step_sizes):
            drift_term = (self.drift - 0.5 * self.volatility**2) * dt
            diffusion_term = (
                self.volatility * np.sqrt(dt) * shocks[step_index]
            )
            out[:, step_index + 1] = out[:, step_index] * np.exp(
                drift_term + diffusion_term
            )
        return out

## Jump Diffusion (Merton)

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class JumpDiffusion:
    drift: float
    volatility: float
    jump_intensity: float
    jump_mean: float
    jump_std: float
    seed: int | None = None
    antithetic: bool = True
    moment_matching: bool = True

    def simulate_paths(self, spot, time_grid, paths):
        diffusion_seed = self.seed
        jump_seed = None if self.seed is None else self.seed + 1
        rng = np.random.default_rng(jump_seed)
        step_sizes = np.diff(time_grid)
        shocks = standard_normals(
            (step_sizes.size, paths),
            seed=diffusion_seed,
            antithetic=self.antithetic,
            moment_matching=self.moment_matching,
        )
        jump_compensation = self.jump_intensity * (
            math.exp(self.jump_mean + 0.5 * self.jump_std**2) - 1.0
        )
        out = np.empty((paths, time_grid.size), dtype=float)
        out[:, 0] = spot
        for step_index, dt in enumerate(step_sizes):
            poisson = rng.poisson(
                self.jump_intensity * float(dt),
                size=paths,
            )
            jump_normals = rng.standard_normal(size=paths)
            sqrt_poisson = np.sqrt(poisson.astype(float, copy=False))
            jump_exponent = (
                poisson * self.jump_mean
                + self.jump_std * sqrt_poisson * jump_normals
            )
            jump_factor = np.exp(jump_exponent)
            drift_term = (
                self.drift
                - jump_compensation
                - 0.5 * self.volatility**2
            ) * float(dt)
            diffusion_term = (
                self.volatility * math.sqrt(float(dt)) * shocks[step_index]
            )
            out[:, step_index + 1] = (
                out[:, step_index]
                * np.exp(drift_term + diffusion_term)
                * jump_factor
            )
        return out

## Heston Stochastic Volatility

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class HestonModel:
    kappa: float
    theta: float
    vol_of_vol: float
    rho: float
    drift: float
    seed: int | None = None
    antithetic: bool = True
    moment_matching: bool = True

    def simulate_paths(self, spot, variance, time_grid, paths):
        step_sizes = np.diff(time_grid)
        normals = standard_normals(
            (step_sizes.size, 2, paths),
            seed=self.seed,
            antithetic=self.antithetic,
            moment_matching=self.moment_matching,
        )
        out_spot = np.empty((paths, time_grid.size), dtype=float)
        out_var = np.empty_like(out_spot)
        out_spot[:, 0] = spot
        out_var[:, 0] = variance

        sqrt_one_minus_rho2 = math.sqrt(1.0 - self.rho**2)
        for step_index, dt in enumerate(step_sizes):
            z1 = normals[step_index, 0, :]
            z2 = normals[step_index, 1, :]
            w1 = z1
            w2 = self.rho * z1 + sqrt_one_minus_rho2 * z2

            prev_var = out_var[:, step_index]
            prev_var_pos = np.maximum(prev_var, 0.0)
            sqrt_var = np.sqrt(prev_var_pos)
            drift_var = (
                self.kappa * (self.theta - prev_var_pos) * float(dt)
            )
            diffusion_var = (
                self.vol_of_vol * sqrt_var * math.sqrt(float(dt)) * w2
            )
            new_var = np.maximum(
                prev_var_pos + drift_var + diffusion_var,
                0.0,
            )
            out_var[:, step_index + 1] = new_var

            prev_spot = out_spot[:, step_index]
            drift_spot = (self.drift - 0.5 * prev_var_pos) * float(dt)
            diffusion_spot = sqrt_var * math.sqrt(float(dt)) * w1
            out_spot[:, step_index + 1] = prev_spot * np.exp(
                drift_spot + diffusion_spot
            )
        return out_spot, out_var

## CIR Short Rate Paths (Extension Point for Stochastic Discounting)

Execute the code examples below.


In [ ]:
@dataclass(frozen=True, slots=True)
class CIRShortRate:
    kappa: float
    theta: float
    sigma: float
    seed: int | None = None

    def simulate_paths(self, rate0, time_grid, paths):
        rng = np.random.default_rng(self.seed)
        step_sizes = np.diff(time_grid)
        out = np.empty((paths, time_grid.size), dtype=float)
        out[:, 0] = rate0
        for step_index, dt in enumerate(step_sizes):
            z = rng.standard_normal(paths)
            prev = out[:, step_index]
            prev_pos = np.maximum(prev, 0.0)
            drift = self.kappa * (self.theta - prev_pos) * float(dt)
            diffusion = (
                self.sigma * np.sqrt(prev_pos) * math.sqrt(float(dt)) * z
            )
            out[:, step_index + 1] = np.maximum(
                prev_pos + drift + diffusion,
                0.0,
            )
        return out

## Diagnostics: Path Plots and Terminal Distributions

Execute the code examples below.


## A Quick Interactive Sanity Check

Execute the code examples below.


In [ ]:
import sys

from pathlib import Path

CODE_DIR = Path("code").resolve()

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


import numpy as np

from dxlib import (
    CIRShortRate,
    GeometricBrownianMotion,
    HestonModel,
    JumpDiffusion,
    build_time_grid,
)

spot = 100.0

maturity = 1.0

steps = 12

paths = 20_000

grid = build_time_grid(maturity=maturity, steps=steps)

seed = 2828

In [ ]:
gbm = GeometricBrownianMotion(
    drift=0.03,
    volatility=0.20,
    seed=seed,
)

gbm_paths = gbm.simulate_paths(
    spot=spot,
    time_grid=grid,
    paths=paths,
)

gbm_paths.shape

In [ ]:
jd = JumpDiffusion(
    drift=0.03,
    volatility=0.18,
    jump_intensity=0.6,
    jump_mean=-0.08,
    jump_std=0.25,
    seed=seed,
)

jd_paths = jd.simulate_paths(
    spot=spot,
    time_grid=grid,
    paths=paths,
)

jd_terminal = jd_paths[:, -1]

p = np.percentile(jd_terminal, [10, 50, 90])

tuple(round(float(x), 4) for x in p)

In [ ]:
heston = HestonModel(
    kappa=2.0,
    theta=0.04,
    vol_of_vol=0.5,
    rho=-0.7,
    drift=0.03,
    seed=seed,
)

hes_spot, hes_var = heston.simulate_paths(
    spot=spot,
    variance=0.04,
    time_grid=grid,
    paths=paths,
)

hes_spot.shape, hes_var.shape

## Where We Are Heading Next

Execute the code examples below.


## Appendix: `dxlib` Simulation Source Code

Execute the code examples below.


## `code/dxlib/random.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 28 - Simulation of Financial Models.

Random number helpers for Monte Carlo simulation.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

from typing import Any

import numpy as np
from numpy.typing import NDArray

__all__ = ["standard_normals"]

FloatArray = NDArray[np.float64]


def standard_normals(
    shape: tuple[int, ...],
    *,
    seed: int | None = None,
    antithetic: bool = False,
    moment_matching: bool = False,
) -> FloatArray:
    """
    Generate standard normal random numbers with optional variance reduction.

    Parameters
    ----------
    shape:
        Output shape of the array.
    seed:
        Seed for reproducibility. ``None`` uses NumPy's default RNG seeding.
    antithetic:
        If ``True``, generate antithetic pairs along the last axis.
        This requires
        the last dimension to be even.
    moment_matching:
        If ``True``, shift and scale the sample so it has mean 0 and standard
        deviation 1.

    Returns
    -------
    ndarray
        Array of standard normal variates.
    """

    if any(dim <= 0 for dim in shape):
        raise ValueError("All shape dimensions must be positive")

    rng = np.random.default_rng(seed=seed)
    out = rng.standard_normal(size=shape).astype(float, copy=False)

    if antithetic:
        if len(shape) == 0:
            raise ValueError(
                "antithetic sampling requires an array, not a scalar"
            )
        last = shape[-1]
        if last % 2 != 0:
            raise ValueError(
                "antithetic sampling requires an even last dimension"
            )
        half = last // 2
        base = out[..., :half]
        out[..., half:] = -base

    if moment_matching:
        mean = float(out.mean())
        std = float(out.std(ddof=0))
        if std == 0.0:
            raise ValueError(
                "moment matching failed: sample standard deviation is zero"
            )
        out = (out - mean) / std

    return out

## `code/dxlib/processes.py`

Execute the code examples below.


In [ ]:
"""Python for Finance, 3rd ed., O'Reilly (2026).
Chapter 28 - Simulation of Financial Models.

Risk-factor simulation models used throughout Part VI.

(c) Dr. Yves J. Hilpisch
AI-supported by various LLMs
The Python Quants GmbH | https://tpq.io
https://hilpisch.com | https://linktr.ee/dyjh
"""

from __future__ import annotations

__package__ = "dxlib"

import math
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import numpy as np
from numpy.typing import NDArray

if __name__ == "__main__" and __package__ is None:
    package_dir = Path(__file__).resolve().parent
    sys.path = [
        path
        for path in sys.path
        if Path(path or ".").resolve() != package_dir
    ]
    sys.path.insert(0, str(package_dir.parent))
    __package__ = "dxlib"

from .random import standard_normals

__all__ = [
    "PathSimulator",
    "build_time_grid",
    "GeometricBrownianMotion",
    "JumpDiffusion",
    "HestonModel",
    "CIRShortRate",
]

FloatArray = NDArray[np.float64]


class PathSimulator(Protocol):
    """
    Protocol for single-factor simulators used by valuation routines.
    """

    def simulate_paths(
        self,
        spot: float,
        time_grid: FloatArray,
        paths: int,
    ) -> FloatArray: ...


def build_time_grid(maturity: float, steps: int) -> FloatArray:
    """
    Build a uniform time grid from 0 to maturity (inclusive).
    """

    if maturity <= 0:
        raise ValueError("maturity must be positive")
    if steps <= 0:
        raise ValueError("steps must be positive")
    grid = np.linspace(0.0, float(maturity), int(steps) + 1)
    return grid.astype(float, copy=False)


def _validate_grid(grid: FloatArray) -> None:
    if grid.ndim != 1:
        raise ValueError("time_grid must be one-dimensional")
    if grid.size < 2:
        raise ValueError("time_grid must contain at least two points")
    if float(grid[0]) != 0.0:
        raise ValueError("time_grid must start at 0")
    if np.any(np.diff(grid) <= 0):
        raise ValueError("time_grid must be strictly increasing")


@dataclass(frozen=True, slots=True)
class GeometricBrownianMotion:
    """
    Geometric Brownian motion (GBM) process for equity prices.

    The process is
    dS_t = mu S_t dt + sigma S_t dW_t,
    and the simulation uses the exact log-normal step.
    """

    drift: float
    volatility: float
    seed: int | None = None
    antithetic: bool = True
    moment_matching: bool = True

    def __post_init__(self) -> None:
        if self.volatility < 0:
            raise ValueError("volatility must be non-negative")

    def simulate_paths(
        self,
        spot: float,
        time_grid: FloatArray,
        paths: int,
    ) -> FloatArray:
        _validate_grid(time_grid)
        if spot <= 0:
            raise ValueError("spot must be positive")
        if paths <= 0:
            raise ValueError("paths must be positive")

        step_sizes = np.diff(time_grid)
        n_steps = step_sizes.size
        shocks = standard_normals(
            (n_steps, paths),
            seed=self.seed,
            antithetic=self.antithetic,
            moment_matching=self.moment_matching,
        )

        out = np.empty((paths, time_grid.size), dtype=float)
        out[:, 0] = float(spot)

        for step_index, dt in enumerate(step_sizes):
            drift_term = (self.drift - 0.5 * self.volatility**2) * dt
            diffusion_term = (
                self.volatility * math.sqrt(float(dt)) * shocks[step_index]
            )
            out[:, step_index + 1] = out[:, step_index] * np.exp(
                drift_term + diffusion_term
            )

        return out


@dataclass(frozen=True, slots=True)
class JumpDiffusion:
    """
    Merton (1976) jump diffusion with log-normal jumps.

    Jump arrivals follow a Poisson process with intensity lambda,
    and log jump sizes are normal.
    """

    drift: float
    volatility: float
    jump_intensity: float
    jump_mean: float
    jump_std: float
    seed: int | None = None
    antithetic: bool = True
    moment_matching: bool = True

    def __post_init__(self) -> None:
        if self.volatility < 0:
            raise ValueError("volatility must be non-negative")
        if self.jump_intensity < 0:
            raise ValueError("jump_intensity must be non-negative")
        if self.jump_std < 0:
            raise ValueError("jump_std must be non-negative")

    def simulate_paths(
        self,
        spot: float,
        time_grid: FloatArray,
        paths: int,
    ) -> FloatArray:
        _validate_grid(time_grid)
        if spot <= 0:
            raise ValueError("spot must be positive")
        if paths <= 0:
            raise ValueError("paths must be positive")

        diffusion_seed = self.seed
        jump_seed = None if self.seed is None else self.seed + 1
        rng = np.random.default_rng(jump_seed)
        step_sizes = np.diff(time_grid)
        n_steps = step_sizes.size
        shocks = standard_normals(
            (n_steps, paths),
            seed=diffusion_seed,
            antithetic=self.antithetic,
            moment_matching=self.moment_matching,
        )

        jump_compensation = self.jump_intensity * (
            math.exp(self.jump_mean + 0.5 * self.jump_std**2) - 1.0
        )

        out = np.empty((paths, time_grid.size), dtype=float)
        out[:, 0] = float(spot)

        for step_index, dt in enumerate(step_sizes):
            poisson = rng.poisson(self.jump_intensity * float(dt), size=paths)
            jump_normals = rng.standard_normal(size=paths)
            if self.antithetic:
                half = paths // 2
                jump_normals[half:] = -jump_normals[:half]

            sqrt_poisson = np.sqrt(poisson.astype(float, copy=False))
            jump_exponent = (
                poisson * self.jump_mean
                + self.jump_std * sqrt_poisson * jump_normals
            )
            jump_factor = np.exp(
                jump_exponent
            )  # product of N iid log-normal jumps

            drift_term = (
                self.drift - jump_compensation - 0.5 * self.volatility**2
            ) * float(dt)
            diffusion_term = (
                self.volatility * math.sqrt(float(dt)) * shocks[step_index]
            )
            out[:, step_index + 1] = (
                out[:, step_index]
                * np.exp(drift_term + diffusion_term)
                * jump_factor
            )

        return out


@dataclass(frozen=True, slots=True)
class HestonModel:
    """
    Heston (1993) stochastic volatility model (Euler-style discretization).

    The model evolves spot S_t and variance v_t
    with correlated Brownian motions.
    """

    kappa: float
    theta: float
    vol_of_vol: float
    rho: float
    drift: float
    seed: int | None = None
    antithetic: bool = True
    moment_matching: bool = True

    def __post_init__(self) -> None:
        if self.kappa <= 0:
            raise ValueError("kappa must be positive")
        if self.theta <= 0:
            raise ValueError("theta must be positive")
        if self.vol_of_vol < 0:
            raise ValueError("vol_of_vol must be non-negative")
        if not -1.0 <= self.rho <= 1.0:
            raise ValueError("rho must be in [-1, 1]")

    def simulate_paths(
        self,
        spot: float,
        variance: float,
        time_grid: FloatArray,
        paths: int,
    ) -> tuple[FloatArray, FloatArray]:
        _validate_grid(time_grid)
        if spot <= 0:
            raise ValueError("spot must be positive")
        if variance <= 0:
            raise ValueError("variance must be positive")
        if paths <= 0:
            raise ValueError("paths must be positive")

        step_sizes = np.diff(time_grid)
        n_steps = step_sizes.size
        normals = standard_normals(
            (n_steps, 2, paths),
            seed=self.seed,
            antithetic=self.antithetic,
            moment_matching=self.moment_matching,
        )

        out_spot = np.empty((paths, time_grid.size), dtype=float)
        out_var = np.empty_like(out_spot)
        out_spot[:, 0] = float(spot)
        out_var[:, 0] = float(variance)

        sqrt_one_minus_rho2 = math.sqrt(1.0 - self.rho**2)

        for step_index, dt in enumerate(step_sizes):
            z1 = normals[step_index, 0, :]
            z2 = normals[step_index, 1, :]
            w1 = z1
            w2 = self.rho * z1 + sqrt_one_minus_rho2 * z2

            prev_var = out_var[:, step_index]
            prev_spot = out_spot[:, step_index]

            prev_var_pos = np.maximum(prev_var, 0.0)
            sqrt_var = np.sqrt(prev_var_pos)

            drift_var = self.kappa * (self.theta - prev_var_pos) * float(dt)
            diffusion_var = (
                self.vol_of_vol * sqrt_var * math.sqrt(float(dt)) * w2
            )
            new_var = np.maximum(prev_var_pos + drift_var + diffusion_var, 0.0)
            out_var[:, step_index + 1] = new_var

            drift_spot = (self.drift - 0.5 * prev_var_pos) * float(dt)
            diffusion_spot = sqrt_var * math.sqrt(float(dt)) * w1
            out_spot[:, step_index + 1] = prev_spot * np.exp(
                drift_spot + diffusion_spot
            )

        return out_spot, out_var


@dataclass(frozen=True, slots=True)
class CIRShortRate:
    """
    Cox-Ingersoll-Ross (1985) short-rate model (Euler discretization with
    truncation).
    """

    kappa: float
    theta: float
    sigma: float
    seed: int | None = None

    def __post_init__(self) -> None:
        if self.kappa <= 0:
            raise ValueError("kappa must be positive")
        if self.theta <= 0:
            raise ValueError("theta must be positive")
        if self.sigma < 0:
            raise ValueError("sigma must be non-negative")

    def simulate_paths(
        self,
        rate0: float,
        time_grid: FloatArray,
        paths: int,
    ) -> FloatArray:
        _validate_grid(time_grid)
        if rate0 < 0:
            raise ValueError("rate0 must be non-negative")
        if paths <= 0:
            raise ValueError("paths must be positive")

        rng = np.random.default_rng(self.seed)
        step_sizes = np.diff(time_grid)
        out = np.empty((paths, time_grid.size), dtype=float)
        out[:, 0] = float(rate0)

        for step_index, dt in enumerate(step_sizes):
            z = rng.standard_normal(paths)
            prev = out[:, step_index]
            prev_pos = np.maximum(prev, 0.0)
            drift = self.kappa * (self.theta - prev_pos) * float(dt)
            diffusion = (
                self.sigma * np.sqrt(prev_pos) * math.sqrt(float(dt)) * z
            )
            out[:, step_index + 1] = np.maximum(
                prev_pos + drift + diffusion,
                0.0,
            )

        return out

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
